# 07 — Process European Commission registry documents

## Purpose

This notebook converts the raw source files downloaded by
`06_download_com_registry.ipynb` into a single reusable text representation for
each local file.

The processing step is deliberately simple and mirrors the Commission legacy and
DB-COMP processors:

1. use the file manifest from notebook 06 as the authoritative handoff;
2. preserve the existing file identifiers, source relationships and raw paths;
3. extract HTML and plain-text files directly;
4. process PDFs with the deterministic fallback
   **PyMuPDF → pdftotext → MinerU OCR**;
5. decide whether a fallback is needed using a small set of transparent technical
   checks rather than a weighted quality score;
6. save one canonical UTF-8 `.txt` file for each processed raw file;
7. record resumable processing state in JSONL.

No Markdown document copies, candidate directories, quality-score tables or
regex/LLM-specific text variants are produced.

The notebook does not reinterpret the legal scope established in notebook 06.
It only processes source files that notebook 06 successfully downloaded.


## Inputs and outputs

### Input

The authoritative file-level input is:

`output/com_antitrust/com_registry_files.csv`

from notebook 06. Only rows with download status `downloaded` or
`already_present` and a local file path enter text processing.

Notebook 06 may associate one physical source file with several Commission
registry rows or decisions. Those relationships are retained through
`document_ids`, `decision_ids`, `case_numbers` and `item_labels`.

### Outputs

Canonical processed text:

`data/processed/com_registry/text/<original_raw_basename>.txt`

The original raw filename is preserved and only its extension is replaced with
`.txt`. For example:

`COMFILE_abcd1234.pdf → COMFILE_abcd1234.txt`

Publication/review metadata:

- `output/com_registry/com_registry_text_manifest.csv`
- `output/com_registry/com_registry_processing.xlsx`

Operational state:

- `logs/com_registry/processing_state.jsonl`

The JSONL file is a resumable processing log and is not part of the analytical
dataset.


In [10]:
from __future__ import annotations

import hashlib
import importlib.metadata
import json
import os
import re
import shutil
import subprocess
import tempfile
import unicodedata
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import fitz  # PyMuPDF
import pandas as pd
from bs4 import BeautifulSoup, UnicodeDammit
from charset_normalizer import from_bytes as charset_from_bytes
from tqdm.auto import tqdm


# ---------------------------------------------------------------------
# Repository paths
# ---------------------------------------------------------------------

REPOSITORY_LABEL = "ec_cjeu"


def find_project_root(start: Path | None = None) -> Path:
    """Locate ec_cjeu from the code/01_commission repository layout."""
    override = os.environ.get("ECCJEU_PROJECT_ROOT", "").strip()
    if override:
        root = Path(override).expanduser().resolve()
        if not root.exists():
            raise FileNotFoundError(
                f"ECCJEU_PROJECT_ROOT does not exist: {root}"
            )
        return root

    start = (start or Path.cwd()).resolve()

    for candidate in [start, *start.parents]:
        if (candidate / "code" / "01_commission").exists():
            return candidate

    for candidate in [start, *start.parents]:
        if (candidate / ".git").exists():
            return candidate

    raise FileNotFoundError(
        "Could not locate the ec_cjeu repository root. Run the notebook "
        "from within the repository or set ECCJEU_PROJECT_ROOT."
    )


PROJECT_ROOT = find_project_root()

# Notebook 06 currently writes its registry data products here.
INPUT_FILES_CSV = (
    PROJECT_ROOT
    / "output"
    / "com_antitrust"
    / "com_registry_files.csv"
)

PROCESSED_TEXT_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "com_registry"
    / "text"
)

OUTPUT_DIR = PROJECT_ROOT / "output" / "com_registry"
LOG_DIR = PROJECT_ROOT / "logs" / "com_registry"

TEXT_MANIFEST_CSV = OUTPUT_DIR / "com_registry_text_manifest.csv"
PROCESSING_XLSX = OUTPUT_DIR / "com_registry_processing.xlsx"
PROCESSING_STATE_JSONL = LOG_DIR / "processing_state.jsonl"

for directory in [PROCESSED_TEXT_DIR, OUTPUT_DIR, LOG_DIR]:
    directory.mkdir(parents=True, exist_ok=True)


# ---------------------------------------------------------------------
# Processing configuration
# ---------------------------------------------------------------------

PROCESSING_VERSION = "1.0"
FORCE_REPROCESS = False

RUN_PDFTOTEXT = True
RUN_OCR = True

PDFTOTEXT_TIMEOUT_SECONDS = 300

# MinerU OCR runtime controls.
# The outer subprocess timeout scales with PDF length, while MinerU's own
# internal timeouts are also raised so they do not terminate long scans first.
OCR_TIMEOUT_SECONDS = 1800          # minimum: 30 minutes
OCR_SECONDS_PER_PAGE = 60
OCR_TIMEOUT_CAP_SECONDS = 7200      # maximum: 2 hours

MINERU_PDF_RENDER_TIMEOUT_SECONDS = 1800
MINERU_LOCAL_API_STARTUP_TIMEOUT_SECONDS = 600
MINERU_TASK_RESULT_TIMEOUT_SECONDS = 7200
MINERU_PROCESSING_WINDOW_SIZE = 32

# MinerU is only called after both native extractors fail their checks.
# Therefore use the general pipeline backend and force OCR explicitly.
MINERU_FORCE_OCR_MODE = True
MINERU_PREFER_PIPELINE_BACKEND = True

# Transparent technical checks used only to decide whether a fallback
# extraction method is needed.
MIN_TOTAL_CHARS = 200
MIN_PDF_CHARS_PER_PAGE = 500
MAX_BAD_ENCODING_MARKERS = 3
MAX_ONE_CHAR_TOKEN_RATIO = 0.20

BAD_ENCODING_MARKERS = [
    "�",
    "Ã",
    "Â",
    "â€™",
    "â€œ",
    "â€",
    "ðŸ",
]

SUPPORTED_FILE_FORMATS = {
    "pdf",
    "html",
    "htm",
    "txt",
    "text",
}

SUCCESSFUL_DOWNLOAD_STATUSES = {
    "downloaded",
    "already_present",
}

PDFTOTEXT_CLI = shutil.which("pdftotext")
MINERU_CLI = shutil.which("mineru") or shutil.which("magic-pdf")


def repo_path(path: Path) -> str:
    """Portable path shown as ec_cjeu/<path inside repository>."""
    relative = (
        path.resolve()
        .relative_to(PROJECT_ROOT.resolve())
        .as_posix()
    )
    return f"{REPOSITORY_LABEL}/{relative}"


def resolve_repo_path(value: str) -> Path:
    """Resolve data/... or ec_cjeu/data/... paths from notebook 06."""
    value = str(value or "").strip().replace("\\", "/")

    if not value:
        raise ValueError("Empty repository path.")

    parts = Path(value).parts
    if parts and parts[0] == REPOSITORY_LABEL:
        value = Path(*parts[1:]).as_posix()

    path = Path(value)

    if (
        path.is_absolute()
        or ".." in path.parts
        or re.match(r"^[A-Za-z]:[/\\]", value)
    ):
        raise ValueError(f"Non-portable repository path: {value}")

    return PROJECT_ROOT / path


def utc_now_iso() -> str:
    return (
        datetime.now(timezone.utc)
        .replace(microsecond=0)
        .isoformat()
    )


def sha256_file(
    path: Path,
    chunk_size: int = 1024 * 1024,
) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)

    return digest.hexdigest()


print("Project root: .")
print(
    "Input file manifest:",
    "output/com_antitrust/com_registry_files.csv",
)
print(
    "Processed text:",
    "data/processed/com_registry/text/",
)
print(
    "Processing manifest:",
    "output/com_registry/com_registry_text_manifest.csv",
)
print(
    "Processing workbook:",
    "output/com_registry/com_registry_processing.xlsx",
)
print(
    "Processing state:",
    "logs/com_registry/processing_state.jsonl",
)
print("pdftotext available:", PDFTOTEXT_CLI is not None)
print("MinerU available:", MINERU_CLI is not None)


Project root: .
Input file manifest: output/com_antitrust/com_registry_files.csv
Processed text: data/processed/com_registry/text/
Processing manifest: output/com_registry/com_registry_text_manifest.csv
Processing workbook: output/com_registry/com_registry_processing.xlsx
Processing state: logs/com_registry/processing_state.jsonl
pdftotext available: True
MinerU available: True


## 1. Validate the handoff from notebook 06

Notebook 07 does not rebuild URLs, file identifiers or decision relationships.
Those belong to the acquisition step.

The handoff checks only the information needed for processing:

- `file_id` is present and unique;
- successful download rows have a local repository-relative path;
- the referenced raw files exist;
- the raw filename remains unique after changing its extension to `.txt`;
- SHA-256 values from notebook 06 are well formed.

The actual raw-file SHA-256 is checked again inside the processing loop before
text extraction or checkpoint reuse. A mismatch is treated as an integrity error
rather than silently processing a changed source file.

Rows that were not downloaded successfully remain the responsibility of notebook 06
and are not duplicated as processing failures here.


In [11]:
REQUIRED_COLUMNS = {
    "file_id",
    "source_url",
    "source_roles",
    "document_ids",
    "decision_ids",
    "case_numbers",
    "item_labels",
    "status",
    "selected_url",
    "relative_path",
    "file_format",
    "sha256",
}


def read_manifest(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(
            f"Required input is missing: {path}"
        )

    return pd.read_csv(
        path,
        dtype=str,
        keep_default_na=False,
    )


files_input = read_manifest(INPUT_FILES_CSV)

missing_columns = sorted(
    REQUIRED_COLUMNS - set(files_input.columns)
)
if missing_columns:
    raise ValueError(
        "com_registry_files.csv is missing required columns: "
        f"{missing_columns}"
    )

if files_input["file_id"].eq("").any():
    raise ValueError(
        "com_registry_files.csv contains an empty file_id."
    )

if not files_input["file_id"].is_unique:
    duplicates = files_input.loc[
        files_input["file_id"].duplicated(keep=False),
        "file_id",
    ].tolist()

    raise ValueError(
        "file_id must be unique; examples: "
        f"{duplicates[:10]}"
    )


available = files_input.loc[
    files_input["status"].isin(SUCCESSFUL_DOWNLOAD_STATUSES)
    & files_input["relative_path"].ne("")
].copy()

if available.empty:
    raise ValueError(
        "No successfully downloaded registry files are available "
        "for processing."
    )

bad_hash = ~available["sha256"].str.fullmatch(
    r"[0-9a-fA-F]{64}"
)
if bad_hash.any():
    examples = available.loc[
        bad_hash,
        ["file_id", "sha256"],
    ].head(10)

    raise ValueError(
        "Successful file rows contain malformed SHA-256 values: "
        f"{examples.to_dict('records')}"
    )


raw_paths = [
    resolve_repo_path(value)
    for value in available["relative_path"]
]

missing_files = [
    available.iloc[index]["file_id"]
    for index, path in enumerate(raw_paths)
    if not path.exists() or not path.is_file()
]

if missing_files:
    raise FileNotFoundError(
        f"{len(missing_files)} referenced raw file(s) are missing; "
        f"examples: {missing_files[:10]}"
    )


def expected_text_name(raw_path: Path) -> str:
    return raw_path.with_suffix(".txt").name


text_names = [
    expected_text_name(path)
    for path in raw_paths
]

if len(text_names) != len(set(text_names)):
    duplicate_names = (
        pd.Series(text_names)
        .value_counts()
        .loc[lambda s: s > 1]
        .index
        .tolist()
    )

    raise ValueError(
        "Changing raw-file extensions to .txt would create duplicate "
        f"processed filenames: {duplicate_names[:10]}"
    )


work_manifest = available.copy()
work_manifest["raw_path_internal"] = [
    str(path)
    for path in raw_paths
]
work_manifest["raw_file_name"] = [
    path.name
    for path in raw_paths
]
work_manifest["text_file_name"] = text_names
work_manifest["raw_sha256"] = (
    work_manifest["sha256"]
    .str.lower()
)

work_manifest = (
    work_manifest
    .sort_values("file_id")
    .reset_index(drop=True)
)

print(f"Registry file rows in notebook 06: {len(files_input):,}")
print(f"Files available for text processing: {len(work_manifest):,}")
print(
    "Rows not processed here:",
    len(files_input) - len(work_manifest),
)

display(
    work_manifest[
        [
            "file_id",
            "raw_file_name",
            "file_format",
            "case_numbers",
            "relative_path",
        ]
    ].head()
)


Registry file rows in notebook 06: 1,213
Files available for text processing: 1,150
Rows not processed here: 63


,file_id,raw_file_name,file_format,case_numbers,relative_path
0,COMFILE_002bcff7abce8b5d,COMFILE_002bcff7abce8b5d.pdf,pdf,AT.39731,data/raw/com_antitrust/COMFILE_002bcff7abce8b5...
1,COMFILE_005ec3e5f1f8142a,COMFILE_005ec3e5f1f8142a.pdf,pdf,AT.39563,data/raw/com_antitrust/COMFILE_005ec3e5f1f8142...
2,COMFILE_00c0de527d9c48b1,COMFILE_00c0de527d9c48b1.pdf,pdf,AT.39824,data/raw/com_antitrust/COMFILE_00c0de527d9c48b...
3,COMFILE_00c8bd19635ca445,COMFILE_00c8bd19635ca445.html,html,AT.39592,data/raw/com_antitrust/COMFILE_00c8bd19635ca44...
4,COMFILE_00f502306c953310,COMFILE_00f502306c953310.html,html,AT.40013,data/raw/com_antitrust/COMFILE_00f502306c95331...


## 2. Canonical text representation

The deposited text is intended to be a general reusable representation rather than
a version optimized for a particular citation parser or language model.

Cleaning is therefore deliberately light:

- Unicode is normalized with NFKC;
- byte-order marks, zero-width spaces and soft hyphens are removed;
- ordinary whitespace and line endings are normalized;
- words broken by line-ending hyphenation are joined;
- line and paragraph structure is otherwise preserved.

Only one `.txt` representation is retained.


In [12]:
def normalize_unicode(text: str) -> str:
    text = unicodedata.normalize("NFKC", text or "")

    replacements = {
        "\u00a0": " ",
        "\u200b": "",
        "\ufeff": "",
        "\r\n": "\n",
        "\r": "\n",
        "\f": "\n\n",
    }

    for old, new in replacements.items():
        text = text.replace(old, new)

    return text


def clean_text(text: str) -> str:
    """Light normalisation while preserving readable structure."""
    text = normalize_unicode(text)
    text = text.replace("\u00ad", "")

    # Join words broken by line-ending hyphenation.
    text = re.sub(
        r"(?<=[A-Za-zÀ-ÖØ-öø-ÿ])[-‐‑]\s*\n\s*(?=[a-zà-öø-ÿ])",
        "",
        text,
    )

    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r" *\n *", "\n", text)
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip() + "\n" if text.strip() else ""


def count_bad_encoding_markers(text: str) -> int:
    return sum(
        text.count(marker)
        for marker in BAD_ENCODING_MARKERS
    )


def one_char_token_ratio(text: str) -> float:
    tokens = re.findall(
        r"[A-Za-zÀ-ÖØ-öø-ÿ]+",
        text or "",
    )

    if not tokens:
        return 1.0

    return (
        sum(len(token) == 1 for token in tokens)
        / len(tokens)
    )


## 3. When is an extraction considered suspicious?

The fallback decision is based only on transparent technical indicators.

A PDF extraction is considered suspicious when any of the following applies:

- fewer than `MIN_TOTAL_CHARS` characters are extracted;
- fewer than `MIN_PDF_CHARS_PER_PAGE` characters are extracted per page;
- at least `MAX_BAD_ENCODING_MARKERS` obvious encoding-corruption markers occur;
- more than `MAX_ONE_CHAR_TOKEN_RATIO` of alphabetic tokens contain a single
  character, which can indicate severely fragmented PDF text.

HTML and plain-text files use the total-length, encoding and fragmentation checks
for review purposes, but OCR is never applied to them.

These checks do **not** use language plausibility, legal-word dictionaries,
candidate scores, duplicate-line scores, selection margins or weighted quality
metrics. The multilingual nature of the Commission registry is therefore not itself
a reason to invoke OCR.


In [13]:
def assess_text(
    text: str,
    *,
    file_format: str,
    n_pages: int | None,
) -> dict[str, Any]:
    """Technical checks used for fallback and review decisions."""
    text = text or ""

    n_chars = len(text)
    bad_marker_count = count_bad_encoding_markers(text)
    fragment_ratio = one_char_token_ratio(text)

    chars_per_page = None
    if file_format == "pdf" and n_pages:
        chars_per_page = n_chars / n_pages

    reasons: list[str] = []

    if n_chars < MIN_TOTAL_CHARS:
        reasons.append("too_little_text")

    if (
        file_format == "pdf"
        and chars_per_page is not None
        and chars_per_page < MIN_PDF_CHARS_PER_PAGE
    ):
        reasons.append("low_text_per_page")

    if bad_marker_count >= MAX_BAD_ENCODING_MARKERS:
        reasons.append("encoding_corruption")

    if fragment_ratio > MAX_ONE_CHAR_TOKEN_RATIO:
        reasons.append("fragmented_text")

    return {
        "passes": not reasons,
        "reasons": reasons,
        "n_chars": n_chars,
        "n_pages": n_pages,
        "chars_per_page": (
            round(chars_per_page, 1)
            if chars_per_page is not None
            else None
        ),
        "bad_encoding_markers": bad_marker_count,
        "one_char_token_ratio": round(
            fragment_ratio,
            4,
        ),
    }


## 4. HTML and plain-text extraction

Commission registry source material is not uniformly distributed as PDF. Some
documents are supplied as HTML.

HTML decoding first respects explicit encoding information and then uses standard
fallback detection. Obvious page chrome such as navigation, scripts, forms and
cookie elements is removed. The largest plausible document container is retained.

This is intentionally simpler than ranking HTML containers with a language-based
quality score.

Plain-text files are decoded using the same encoding logic and then lightly
normalized.


In [14]:
HTML_REMOVE_SELECTORS = [
    "script",
    "style",
    "noscript",
    "nav",
    "footer",
    "header",
    "form",
    "aside",
    "[class*='cookie']",
    "[id*='cookie']",
    "[class*='share']",
    "[class*='language']",
    "[aria-label*='language']",
    "[class*='pagination']",
    "[class*='print']",
]

HTML_CANDIDATE_SELECTORS = [
    "main",
    "article",
    "#document1",
    "#TexteOnly",
    "#text",
    ".document-content",
    ".document",
    ".content",
    "body",
]


def decode_issue_count(text: str) -> int:
    control_chars = sum(
        1
        for char in text
        if (
            unicodedata.category(char).startswith("C")
            and char not in "\n\t"
        )
    )

    return (
        count_bad_encoding_markers(text)
        + control_chars
    )


def extract_declared_html_encoding(
    data: bytes,
) -> str | None:
    head = data[:8192].decode(
        "ascii",
        errors="ignore",
    )

    patterns = [
        r'<meta[^>]+charset\s*=\s*["\']?\s*([A-Za-z0-9._-]+)',
        r'<meta[^>]+content\s*=\s*["\'][^"\']*charset\s*=\s*([A-Za-z0-9._-]+)',
        r'<\?xml[^>]+encoding\s*=\s*["\']([^"\']+)',
    ]

    for pattern in patterns:
        match = re.search(
            pattern,
            head,
            flags=re.I,
        )
        if match:
            return match.group(1).strip()

    return None


def decode_bytes_smart(
    data: bytes,
    *,
    is_html: bool = False,
) -> tuple[str, dict[str, Any]]:
    candidates: list[tuple[str, str, str]] = []

    if data.startswith(b"\xef\xbb\xbf"):
        candidates.append(
            (
                "utf-8-sig",
                "bom",
                data.decode(
                    "utf-8-sig",
                    errors="replace",
                ),
            )
        )

    elif data.startswith((b"\xff\xfe", b"\xfe\xff")):
        candidates.append(
            (
                "utf-16",
                "bom",
                data.decode(
                    "utf-16",
                    errors="replace",
                ),
            )
        )

    declared = (
        extract_declared_html_encoding(data)
        if is_html
        else None
    )

    if declared:
        try:
            candidates.append(
                (
                    declared,
                    "declared",
                    data.decode(
                        declared,
                        errors="replace",
                    ),
                )
            )
        except (LookupError, UnicodeError):
            pass

    try:
        candidates.append(
            (
                "utf-8",
                "strict_utf8",
                data.decode(
                    "utf-8",
                    errors="strict",
                ),
            )
        )
    except UnicodeDecodeError:
        pass

    if is_html:
        dammit = UnicodeDammit(
            data,
            is_html=True,
            smart_quotes_to=None,
        )

        if dammit.unicode_markup:
            candidates.append(
                (
                    dammit.original_encoding or "unknown",
                    "unicode_dammit",
                    dammit.unicode_markup,
                )
            )

    try:
        for match in list(
            charset_from_bytes(data)
        )[:3]:
            candidates.append(
                (
                    getattr(
                        match,
                        "encoding",
                        None,
                    ) or "unknown",
                    "charset_normalizer",
                    str(match),
                )
            )
    except (ValueError, TypeError):
        pass

    for encoding in ("cp1252", "latin-1"):
        candidates.append(
            (
                encoding,
                "fallback",
                data.decode(
                    encoding,
                    errors="replace",
                ),
            )
        )

    unique: list[tuple[str, str, str]] = []
    seen = set()

    for encoding, source, text in candidates:
        digest = hashlib.sha1(
            text.encode(
                "utf-8",
                errors="ignore",
            )
        ).hexdigest()

        if digest not in seen:
            seen.add(digest)
            unique.append(
                (
                    encoding,
                    source,
                    text,
                )
            )

    encoding, source, text = min(
        unique,
        key=lambda item: decode_issue_count(
            item[2]
        ),
    )

    return normalize_unicode(text), {
        "encoding": encoding,
        "encoding_source": source,
    }


def extract_text_from_html(
    path: Path,
) -> tuple[str, dict[str, Any]]:
    html, decode_meta = decode_bytes_smart(
        path.read_bytes(),
        is_html=True,
    )

    soup = BeautifulSoup(
        html,
        "html.parser",
    )

    for selector in HTML_REMOVE_SELECTORS:
        for tag in soup.select(selector):
            tag.decompose()

    candidates: list[tuple[int, str, str]] = []

    for selector in HTML_CANDIDATE_SELECTORS:
        for tag in soup.select(selector):
            text = tag.get_text(
                "\n",
                strip=True,
            )

            if len(text) >= 100:
                candidates.append(
                    (
                        len(text),
                        selector,
                        text,
                    )
                )

    if candidates:
        _, selected_selector, text = max(
            candidates,
            key=lambda item: item[0],
        )
    else:
        selected_selector = "document"
        text = soup.get_text(
            "\n",
            strip=True,
        )

    return clean_text(text), {
        "method": "html",
        "n_pages": None,
        "html_selected_container": selected_selector,
        **decode_meta,
    }


def extract_text_from_plain(
    path: Path,
) -> tuple[str, dict[str, Any]]:
    text, decode_meta = decode_bytes_smart(
        path.read_bytes(),
        is_html=False,
    )

    return clean_text(text), {
        "method": "plain_text",
        "n_pages": None,
        **decode_meta,
    }


## 5. PDF extraction

PDFs follow the same deterministic extraction ladder used in the cleaned Commission
processors:

1. **PyMuPDF sorted text** is the default native extraction;
2. suspicious output is retried with **Poppler `pdftotext -layout`**;
3. if both native methods remain suspicious, **MinerU** is used explicitly with the `pipeline` backend and OCR mode.

The methods are not scored against one another. The first extraction satisfying the
predefined technical checks is retained.

MinerU may create Markdown or text internally inside a temporary working directory.
Those temporary files are deleted after extraction. Notebook 07 persists **only**
the final `.txt` representation.


In [15]:
def extract_pdf_pymupdf(
    path: Path,
) -> tuple[str, int | None]:
    page_text: list[str] = []

    with fitz.open(path) as document:
        for page in document:
            page_text.append(
                page.get_text(
                    "text",
                    sort=True,
                ) or ""
            )

        n_pages = len(document)

    return (
        clean_text(
            "\n\n".join(page_text)
        ),
        n_pages,
    )


def extract_pdf_pdftotext(
    path: Path,
) -> tuple[str, int | None]:
    if PDFTOTEXT_CLI is None:
        raise RuntimeError(
            "pdftotext is not available on PATH."
        )

    with tempfile.TemporaryDirectory(
        prefix="pdftotext_"
    ) as tmpdir:
        output_path = (
            Path(tmpdir)
            / "output.txt"
        )

        command = [
            PDFTOTEXT_CLI,
            "-layout",
            "-enc",
            "UTF-8",
            str(path),
            str(output_path),
        ]

        completed = subprocess.run(
            command,
            capture_output=True,
            text=True,
            timeout=PDFTOTEXT_TIMEOUT_SECONDS,
            check=False,
        )

        if completed.returncode != 0:
            raise RuntimeError(
                completed.stderr[-2000:]
                or "pdftotext failed"
            )

        text = output_path.read_text(
            encoding="utf-8",
            errors="replace",
        )

    try:
        with fitz.open(path) as document:
            n_pages = len(document)
    except Exception:
        n_pages = None

    return clean_text(text), n_pages


def run_mineru(
    input_pdf: Path,
    output_dir: Path,
) -> None:
    if MINERU_CLI is None:
        raise RuntimeError(
            "MinerU is not available on PATH."
        )

    try:
        with fitz.open(input_pdf) as document:
            n_pages = len(document)
    except Exception:
        n_pages = None

    if n_pages is None:
        timeout_seconds = OCR_TIMEOUT_SECONDS
    else:
        timeout_seconds = min(
            OCR_TIMEOUT_CAP_SECONDS,
            max(
                OCR_TIMEOUT_SECONDS,
                n_pages * OCR_SECONDS_PER_PAGE,
            ),
        )

    command = [
        MINERU_CLI,
        "-p",
        str(input_pdf),
        "-o",
        str(output_dir),
    ]

    if MINERU_PREFER_PIPELINE_BACKEND:
        command.extend(
            ["-b", "pipeline"]
        )

    if MINERU_FORCE_OCR_MODE:
        command.extend(
            ["-m", "ocr"]
        )

    environment = os.environ.copy()
    environment.setdefault(
        "ORT_LOG_SEVERITY_LEVEL",
        "3",
    )

    # MinerU has internal limits in addition to Python's subprocess timeout.
    # Raise those limits for long image-only PDFs.
    environment[
        "MINERU_PDF_RENDER_TIMEOUT"
    ] = str(
        MINERU_PDF_RENDER_TIMEOUT_SECONDS
    )
    environment[
        "MINERU_LOCAL_API_STARTUP_TIMEOUT_SECONDS"
    ] = str(
        MINERU_LOCAL_API_STARTUP_TIMEOUT_SECONDS
    )
    environment[
        "MINERU_TASK_RESULT_TIMEOUT_SECONDS"
    ] = str(
        MINERU_TASK_RESULT_TIMEOUT_SECONDS
    )
    environment[
        "MINERU_PROCESSING_WINDOW_SIZE"
    ] = str(
        MINERU_PROCESSING_WINDOW_SIZE
    )

    try:
        completed = subprocess.run(
            command,
            capture_output=True,
            text=True,
            timeout=timeout_seconds,
            env=environment,
            check=False,
        )
    except subprocess.TimeoutExpired as exc:
        page_label = (
            n_pages
            if n_pages is not None
            else "unknown"
        )
        raise RuntimeError(
            f"MinerU timed out after "
            f"{timeout_seconds} seconds "
            f"for a {page_label}-page PDF."
        ) from exc

    if completed.returncode != 0:
        stderr_tail = (
            completed.stderr or ""
        )[-4000:].strip()
        stdout_tail = (
            completed.stdout or ""
        )[-2000:].strip()

        detail = (
            stderr_tail
            or stdout_tail
            or "MinerU failed without output."
        )

        raise RuntimeError(
            f"MinerU failed with return code "
            f"{completed.returncode}. "
            f"Pages: "
            f"{n_pages if n_pages is not None else 'unknown'}. "
            f"Timeout allowance: "
            f"{timeout_seconds}s. "
            f"Output tail: {detail}"
        )

def find_mineru_text(
    output_dir: Path,
) -> Path | None:
    # MinerU often writes Markdown as its richest text output.
    # It is used only inside the temporary OCR directory.
    markdown = [
        path
        for path in output_dir.glob("**/*.md")
        if (
            path.is_file()
            and path.stat().st_size > 0
        )
    ]

    if markdown:
        return max(
            markdown,
            key=lambda path: path.stat().st_size,
        )

    text_files = [
        path
        for path in output_dir.glob("**/*.txt")
        if (
            path.is_file()
            and path.stat().st_size > 0
        )
    ]

    if text_files:
        return max(
            text_files,
            key=lambda path: path.stat().st_size,
        )

    return None


def extract_pdf_mineru(
    path: Path,
) -> tuple[str, int | None]:
    with tempfile.TemporaryDirectory(
        prefix="mineru_"
    ) as tmpdir:
        output_dir = Path(tmpdir)
        run_mineru(
            path,
            output_dir,
        )

        extracted_path = find_mineru_text(
            output_dir
        )

        if extracted_path is None:
            raise RuntimeError(
                "MinerU completed but produced no "
                "non-empty text representation."
            )

        text = extracted_path.read_text(
            encoding="utf-8",
            errors="replace",
        )

    try:
        with fitz.open(path) as document:
            n_pages = len(document)
    except Exception:
        n_pages = None

    return clean_text(text), n_pages


def try_pdf_extractor(
    path: Path,
    method: str,
    extractor,
) -> tuple[dict[str, Any] | None, str]:
    try:
        text, n_pages = extractor(path)

        assessment = assess_text(
            text,
            file_format="pdf",
            n_pages=n_pages,
        )

        return {
            "method": method,
            "text": text,
            "assessment": assessment,
        }, ""

    except Exception as exc:
        return None, repr(exc)


## 6. Deterministic PDF fallback

The fallback sequence is evaluated in order.

- If PyMuPDF passes the checks, it is retained.
- Otherwise `pdftotext` is attempted.
- If `pdftotext` also remains suspicious, MinerU OCR is attempted.
- If MinerU produces usable text but one or more simple thresholds still trigger,
  the text is retained with `processing_status = review`.
- If OCR is unavailable or fails, the latest non-empty native extraction is retained
  with `processing_status = review`.
- `failed` is reserved for files from which no usable text can be obtained.

This distinction prevents imperfect but recoverable historical documents from being
silently discarded.


In [16]:
def process_pdf(path: Path) -> dict[str, Any]:
    attempts: list[dict[str, Any]] = []
    errors: list[str] = []
    fallback_reason: list[str] = []

    # --------------------------------------------------------------
    # 1. Default native extraction: PyMuPDF
    # --------------------------------------------------------------

    pymupdf_result, error = try_pdf_extractor(
        path,
        "pymupdf",
        extract_pdf_pymupdf,
    )

    if error:
        errors.append(
            f"pymupdf: {error}"
        )
        fallback_reason.append(
            "pymupdf_failed"
        )

    elif pymupdf_result is not None:
        attempts.append(
            pymupdf_result
        )

        if pymupdf_result[
            "assessment"
        ]["passes"]:
            return {
                "selected": pymupdf_result,
                "status": "success",
                "review_reason": "",
                "ocr_used": False,
                "fallback_reason": "",
                "errors": errors,
            }

        fallback_reason.extend(
            f"pymupdf:{reason}"
            for reason in pymupdf_result[
                "assessment"
            ]["reasons"]
        )

    # --------------------------------------------------------------
    # 2. Native fallback: pdftotext -layout
    # --------------------------------------------------------------

    if (
        RUN_PDFTOTEXT
        and PDFTOTEXT_CLI is not None
    ):
        poppler_result, error = try_pdf_extractor(
            path,
            "pdftotext",
            extract_pdf_pdftotext,
        )

        if error:
            errors.append(
                f"pdftotext: {error}"
            )
            fallback_reason.append(
                "pdftotext_failed"
            )

        elif poppler_result is not None:
            attempts.append(
                poppler_result
            )

            if poppler_result[
                "assessment"
            ]["passes"]:
                return {
                    "selected": poppler_result,
                    "status": "success",
                    "review_reason": "",
                    "ocr_used": False,
                    "fallback_reason": "; ".join(
                        fallback_reason
                    ),
                    "errors": errors,
                }

            fallback_reason.extend(
                f"pdftotext:{reason}"
                for reason in poppler_result[
                    "assessment"
                ]["reasons"]
            )

    elif RUN_PDFTOTEXT:
        fallback_reason.append(
            "pdftotext_unavailable"
        )

    # --------------------------------------------------------------
    # 3. OCR fallback: MinerU
    # --------------------------------------------------------------

    if (
        RUN_OCR
        and MINERU_CLI is not None
    ):
        ocr_result, error = try_pdf_extractor(
            path,
            "mineru_ocr",
            extract_pdf_mineru,
        )

        if error:
            errors.append(
                f"mineru_ocr: {error}"
            )
            fallback_reason.append(
                "mineru_failed"
            )

        elif (
            ocr_result is not None
            and ocr_result["text"].strip()
        ):
            attempts.append(
                ocr_result
            )

            if ocr_result[
                "assessment"
            ]["passes"]:
                return {
                    "selected": ocr_result,
                    "status": "success",
                    "review_reason": "",
                    "ocr_used": True,
                    "fallback_reason": "; ".join(
                        fallback_reason
                    ),
                    "errors": errors,
                }

            return {
                "selected": ocr_result,
                "status": "review",
                "review_reason": "; ".join(
                    f"mineru:{reason}"
                    for reason in ocr_result[
                        "assessment"
                    ]["reasons"]
                ),
                "ocr_used": True,
                "fallback_reason": "; ".join(
                    fallback_reason
                ),
                "errors": errors,
            }

    elif RUN_OCR:
        fallback_reason.append(
            "mineru_unavailable"
        )

    # --------------------------------------------------------------
    # No method passed. Retain the latest non-empty native result.
    # --------------------------------------------------------------

    nonempty_attempts = [
        attempt
        for attempt in attempts
        if attempt.get(
            "text",
            "",
        ).strip()
    ]

    if nonempty_attempts:
        selected = nonempty_attempts[-1]

        return {
            "selected": selected,
            "status": "review",
            "review_reason": "; ".join(
                f"{selected['method']}:{reason}"
                for reason in selected[
                    "assessment"
                ]["reasons"]
            ) or "no_method_passed_checks",
            "ocr_used": (
                selected["method"]
                == "mineru_ocr"
            ),
            "fallback_reason": "; ".join(
                fallback_reason
            ),
            "errors": errors,
        }

    return {
        "selected": None,
        "status": "failed",
        "review_reason": "no_text_extracted",
        "ocr_used": False,
        "fallback_reason": "; ".join(
            fallback_reason
        ),
        "errors": errors,
    }


def process_non_pdf(
    path: Path,
    file_format: str,
) -> dict[str, Any]:
    extractor = (
        extract_text_from_html
        if file_format in {"html", "htm"}
        else extract_text_from_plain
    )

    try:
        text, meta = extractor(path)

        assessment = assess_text(
            text,
            file_format=file_format,
            n_pages=None,
        )

    except Exception as exc:
        return {
            "selected": None,
            "status": "failed",
            "review_reason": "extraction_failed",
            "ocr_used": False,
            "fallback_reason": "",
            "errors": [repr(exc)],
        }

    selected = {
        "method": meta["method"],
        "text": text,
        "assessment": assessment,
    }

    if assessment["passes"]:
        status = "success"
        review_reason = ""
    else:
        status = "review"
        review_reason = "; ".join(
            assessment["reasons"]
        )

    return {
        "selected": selected,
        "status": status,
        "review_reason": review_reason,
        "ocr_used": False,
        "fallback_reason": "",
        "errors": [],
    }


## 7. Resumable JSONL processing state

`logs/com_registry/processing_state.jsonl` is an append-only operational checkpoint.

Each completed processing attempt records the source `file_id`, raw SHA-256,
processing version, selected extraction method, output path and output SHA-256.

On restart, the latest JSONL record for each `file_id` is used. A previous
`success` or `review` result is reused only when:

- the processing version is unchanged;
- the raw SHA-256 still matches notebook 06;
- the expected `.txt` output exists;
- the text SHA-256 still matches the checkpoint.

Failed rows are retried automatically. Set `FORCE_REPROCESS=True` to rebuild every
file deliberately.

This keeps operational history out of the publication manifest while allowing long
OCR runs to resume safely.


In [17]:
def json_safe(value: Any) -> Any:
    if value is None:
        return None

    if isinstance(
        value,
        (str, int, float, bool),
    ):
        if (
            isinstance(value, float)
            and pd.isna(value)
        ):
            return None

        return value

    if isinstance(value, Path):
        return value.as_posix()

    if isinstance(value, dict):
        return {
            str(key): json_safe(item)
            for key, item in value.items()
        }

    if isinstance(
        value,
        (list, tuple, set),
    ):
        return [
            json_safe(item)
            for item in value
        ]

    try:
        if pd.isna(value):
            return None
    except Exception:
        pass

    return str(value)


def load_processing_state() -> dict[str, dict[str, Any]]:
    """
    Keep the latest valid JSONL record for each file_id.

    A malformed final/partial line is ignored so an interrupted write
    does not prevent restart.
    """
    latest: dict[str, dict[str, Any]] = {}

    if not PROCESSING_STATE_JSONL.exists():
        return latest

    with PROCESSING_STATE_JSONL.open(
        "r",
        encoding="utf-8",
    ) as handle:
        for line in handle:
            line = line.strip()

            if not line:
                continue

            try:
                record = json.loads(line)
            except json.JSONDecodeError:
                continue

            file_id = str(
                record.get(
                    "file_id",
                    "",
                )
            ).strip()

            if file_id:
                latest[file_id] = record

    return latest


def append_processing_state(
    record: dict[str, Any],
) -> None:
    with PROCESSING_STATE_JSONL.open(
        "a",
        encoding="utf-8",
    ) as handle:
        handle.write(
            json.dumps(
                json_safe(record),
                ensure_ascii=False,
                sort_keys=True,
            )
            + "\n"
        )


def expected_text_path(
    raw_path: Path,
) -> Path:
    return (
        PROCESSED_TEXT_DIR
        / raw_path.with_suffix(".txt").name
    )


def state_is_reusable(
    state_record: dict[str, Any] | None,
    *,
    raw_sha256: str,
    text_path: Path,
) -> bool:
    if FORCE_REPROCESS or not state_record:
        return False

    if state_record.get(
        "processing_status"
    ) not in {"success", "review"}:
        return False

    if (
        state_record.get("processing_version")
        != PROCESSING_VERSION
    ):
        return False

    if (
        str(
            state_record.get(
                "raw_sha256",
                "",
            )
        ).lower()
        != raw_sha256.lower()
    ):
        return False

    if (
        state_record.get("text_path")
        != repo_path(text_path)
    ):
        return False

    if (
        not text_path.exists()
        or text_path.stat().st_size <= 0
    ):
        return False

    text_sha256 = str(
        state_record.get(
            "text_sha256",
            "",
        )
    ).lower()

    if not re.fullmatch(
        r"[0-9a-f]{64}",
        text_sha256,
    ):
        return False

    return (
        sha256_file(text_path)
        == text_sha256
    )


def process_file(
    input_row: dict[str, Any],
    *,
    raw_sha256: str,
) -> dict[str, Any]:
    file_id = str(
        input_row["file_id"]
    )
    raw_path = Path(
        input_row["raw_path_internal"]
    )
    file_format = str(
        input_row["file_format"]
    ).strip().lower()

    if not file_format:
        file_format = (
            raw_path.suffix
            .lower()
            .lstrip(".")
        )

    text_path = expected_text_path(
        raw_path
    )

    base = {
        "timestamp_utc": utc_now_iso(),
        "processing_version": PROCESSING_VERSION,
        "file_id": file_id,
        "raw_file_name": raw_path.name,
        "raw_path": repo_path(raw_path),
        "raw_sha256": raw_sha256,
    }

    if file_format not in SUPPORTED_FILE_FORMATS:
        return {
            **base,
            "processing_status": "failed",
            "extraction_method": "",
            "ocr_used": False,
            "fallback_reason": "",
            "review_reason": "unsupported_file_format",
            "text_path": "",
            "text_sha256": "",
            "n_pages": None,
            "n_chars": 0,
            "chars_per_page": None,
            "error": (
                "unsupported_file_format:"
                f"{file_format}"
            ),
        }

    result = (
        process_pdf(raw_path)
        if file_format == "pdf"
        else process_non_pdf(
            raw_path,
            file_format,
        )
    )

    selected = result["selected"]

    if selected is None:
        return {
            **base,
            "processing_status": "failed",
            "extraction_method": "",
            "ocr_used": False,
            "fallback_reason": result.get(
                "fallback_reason",
                "",
            ),
            "review_reason": result.get(
                "review_reason",
                "no_text_extracted",
            ),
            "text_path": "",
            "text_sha256": "",
            "n_pages": None,
            "n_chars": 0,
            "chars_per_page": None,
            "error": (
                " | ".join(
                    result.get(
                        "errors",
                        [],
                    )
                )
                or "no_text_extracted"
            ),
        }

    text = selected["text"]
    text_path.write_text(
        text,
        encoding="utf-8",
    )

    assessment = selected[
        "assessment"
    ]

    return {
        **base,
        "processing_status": result["status"],
        "extraction_method": selected["method"],
        "ocr_used": bool(
            result["ocr_used"]
        ),
        "fallback_reason": result.get(
            "fallback_reason",
            "",
        ),
        "review_reason": result.get(
            "review_reason",
            "",
        ),
        "text_path": repo_path(text_path),
        "text_sha256": sha256_file(
            text_path
        ),
        "n_pages": assessment[
            "n_pages"
        ],
        "n_chars": assessment[
            "n_chars"
        ],
        "chars_per_page": assessment[
            "chars_per_page"
        ],
        "error": " | ".join(
            result.get(
                "errors",
                [],
            )
        ),
    }


## 8. Process the registry collection

Each local source file is handled once.

Before extraction or checkpoint reuse, its current SHA-256 is compared with the
SHA-256 recorded by notebook 06. This confirms that the processor is working on the
same raw bytes that were inventoried during acquisition.

The progress bar reports:

- successful files;
- review files;
- failures;
- selected OCR outputs;
- checkpoint rows reused during the current run.


In [18]:
processing_state = load_processing_state()
manifest_rows: list[dict[str, Any]] = []

run_counts = {
    "success": 0,
    "review": 0,
    "failed": 0,
    "ocr": 0,
    "reused": 0,
}

target_rows = work_manifest.to_dict(
    "records"
)

progress = tqdm(
    target_rows,
    desc="Processing COM registry files",
    unit="file",
    dynamic_ncols=True,
)

for input_row in progress:
    file_id = str(
        input_row["file_id"]
    )
    raw_path = Path(
        input_row["raw_path_internal"]
    )

    actual_raw_sha256 = sha256_file(
        raw_path
    )
    expected_raw_sha256 = str(
        input_row["raw_sha256"]
    ).lower()

    if (
        actual_raw_sha256
        != expected_raw_sha256
    ):
        raise ValueError(
            "Raw SHA-256 mismatch for "
            f"{file_id}: {repo_path(raw_path)}"
        )

    text_path = expected_text_path(
        raw_path
    )

    previous_state = processing_state.get(
        file_id
    )

    if state_is_reusable(
        previous_state,
        raw_sha256=actual_raw_sha256,
        text_path=text_path,
    ):
        state_record = previous_state
        run_counts["reused"] += 1

    else:
        state_record = process_file(
            input_row,
            raw_sha256=actual_raw_sha256,
        )

        append_processing_state(
            state_record
        )
        processing_state[
            file_id
        ] = state_record

    manifest_rows.append({
        "file_id": input_row["file_id"],
        "raw_file_name": input_row["raw_file_name"],
        "document_ids": input_row["document_ids"],
        "decision_ids": input_row["decision_ids"],
        "case_numbers": input_row["case_numbers"],
        "item_labels": input_row["item_labels"],
        "source_roles": input_row["source_roles"],
        "source_url": input_row["source_url"],
        "selected_url": input_row["selected_url"],
        "file_format": input_row["file_format"],
        "raw_path": repo_path(raw_path),
        "text_path": state_record.get(
            "text_path",
            "",
        ),
        "extraction_method": state_record.get(
            "extraction_method",
            "",
        ),
        "ocr_used": state_record.get(
            "ocr_used",
            False,
        ),
        "processing_status": state_record.get(
            "processing_status",
            "",
        ),
        "review_reason": state_record.get(
            "review_reason",
            "",
        ),
        "fallback_reason": state_record.get(
            "fallback_reason",
            "",
        ),
        "raw_sha256": actual_raw_sha256,
        "text_sha256": state_record.get(
            "text_sha256",
            "",
        ),
        "n_pages": state_record.get(
            "n_pages",
        ),
        "n_chars": state_record.get(
            "n_chars",
        ),
        "chars_per_page": state_record.get(
            "chars_per_page",
        ),
        "error": state_record.get(
            "error",
            "",
        ),
    })

    status = state_record.get(
        "processing_status",
        "failed",
    )

    if status in {
        "success",
        "review",
        "failed",
    }:
        run_counts[status] += 1
    else:
        run_counts["failed"] += 1

    if state_record.get("ocr_used"):
        run_counts["ocr"] += 1

    progress.set_postfix(
        {
            "ok": run_counts["success"],
            "review": run_counts["review"],
            "fail": run_counts["failed"],
            "ocr": run_counts["ocr"],
            "reused": run_counts["reused"],
        },
        refresh=False,
    )


processing_manifest = (
    pd.DataFrame(manifest_rows)
    .sort_values("file_id")
    .reset_index(drop=True)
)

print(f"Files considered: {len(processing_manifest):,}")
print(
    "Successful:",
    int(
        processing_manifest[
            "processing_status"
        ].eq("success").sum()
    ),
)
print(
    "Review:",
    int(
        processing_manifest[
            "processing_status"
        ].eq("review").sum()
    ),
)
print(
    "Failed:",
    int(
        processing_manifest[
            "processing_status"
        ].eq("failed").sum()
    ),
)
print(
    "OCR selected:",
    int(
        processing_manifest[
            "ocr_used"
        ].fillna(False).astype(bool).sum()
    ),
)
print(
    "Checkpoints reused this run:",
    run_counts["reused"],
)


Processing COM registry files:   0%|          | 0/1150 [00:00<?, ?file/s]

Files considered: 1,150
Successful: 1141
Review: 9
Failed: 0
OCR selected: 92
Checkpoints reused this run: 1128


## 9. Publication and review outputs

The public processing manifest is intentionally compact. It links each raw source
file to its canonical `.txt` file and records only the processing information useful
for provenance or review.

The Excel workbook contains three sheets:

- `documents` — the complete file-level processing manifest;
- `review` — only `review` and `failed` rows;
- `summary` — counts by processing status/method together with the fixed extraction
  thresholds and software versions.

There is no separate candidate manifest, audit CSV, validation CSV or candidate-text
directory. Hard integrity failures stop the notebook; text-quality exceptions are
represented directly by `processing_status` and `review_reason`.


In [19]:
processing_manifest.to_csv(
    TEXT_MANIFEST_CSV,
    index=False,
    encoding="utf-8",
)

review_df = processing_manifest.loc[
    processing_manifest[
        "processing_status"
    ].isin(
        ["review", "failed"]
    )
].copy()


status_counts = (
    processing_manifest[
        "processing_status"
    ]
    .value_counts(dropna=False)
    .rename_axis("value")
    .reset_index(name="files")
)
status_counts.insert(
    0,
    "category",
    "processing_status",
)


method_counts = (
    processing_manifest[
        "extraction_method"
    ]
    .replace("", "(none)")
    .value_counts(dropna=False)
    .rename_axis("value")
    .reset_index(name="files")
)
method_counts.insert(
    0,
    "category",
    "extraction_method",
)


format_counts = (
    processing_manifest[
        "file_format"
    ]
    .replace("", "(unknown)")
    .value_counts(dropna=False)
    .rename_axis("value")
    .reset_index(name="files")
)
format_counts.insert(
    0,
    "category",
    "file_format",
)


def package_version(name: str) -> str:
    try:
        return importlib.metadata.version(
            name
        )
    except importlib.metadata.PackageNotFoundError:
        return "(not installed)"


summary_rows = pd.DataFrame(
    [
        [
            "configuration",
            "processing_version",
            PROCESSING_VERSION,
        ],
        [
            "configuration",
            "min_total_chars",
            MIN_TOTAL_CHARS,
        ],
        [
            "configuration",
            "min_pdf_chars_per_page",
            MIN_PDF_CHARS_PER_PAGE,
        ],
        [
            "configuration",
            "max_bad_encoding_markers",
            MAX_BAD_ENCODING_MARKERS,
        ],
        [
            "configuration",
            "max_one_char_token_ratio",
            MAX_ONE_CHAR_TOKEN_RATIO,
        ],
        [
            "configuration",
            "run_pdftotext",
            RUN_PDFTOTEXT,
        ],
        [
            "configuration",
            "run_ocr",
            RUN_OCR,
        ],
        [
            "configuration",
            "ocr_timeout_min_seconds",
            OCR_TIMEOUT_SECONDS,
        ],
        [
            "configuration",
            "ocr_seconds_per_page",
            OCR_SECONDS_PER_PAGE,
        ],
        [
            "configuration",
            "ocr_timeout_cap_seconds",
            OCR_TIMEOUT_CAP_SECONDS,
        ],
        [
            "configuration",
            "mineru_backend",
            "pipeline" if MINERU_PREFER_PIPELINE_BACKEND else "default",
        ],
        [
            "configuration",
            "mineru_method",
            "ocr" if MINERU_FORCE_OCR_MODE else "auto",
        ],
        [
            "configuration",
            "mineru_processing_window_size",
            MINERU_PROCESSING_WINDOW_SIZE,
        ],
        [
            "software",
            "pandas",
            package_version("pandas"),
        ],
        [
            "software",
            "PyMuPDF",
            package_version("PyMuPDF"),
        ],
        [
            "software",
            "beautifulsoup4",
            package_version("beautifulsoup4"),
        ],
        [
            "software",
            "charset-normalizer",
            package_version("charset-normalizer"),
        ],
    ],
    columns=[
        "category",
        "value",
        "files",
    ],
)


processing_summary = pd.concat(
    [
        status_counts,
        method_counts,
        format_counts,
        summary_rows,
    ],
    ignore_index=True,
)


with pd.ExcelWriter(
    PROCESSING_XLSX,
    engine="openpyxl",
) as writer:
    processing_manifest.to_excel(
        writer,
        sheet_name="documents",
        index=False,
    )

    review_df.to_excel(
        writer,
        sheet_name="review",
        index=False,
    )

    processing_summary.to_excel(
        writer,
        sheet_name="summary",
        index=False,
    )


from openpyxl import load_workbook
from openpyxl.styles import Alignment, Font

workbook = load_workbook(
    PROCESSING_XLSX
)

for worksheet in workbook.worksheets:
    worksheet.freeze_panes = "A2"

    if (
        worksheet.max_row >= 1
        and worksheet.max_column >= 1
    ):
        worksheet.auto_filter.ref = (
            worksheet.dimensions
        )

        for cell in worksheet[1]:
            cell.font = Font(
                bold=True
            )

    for column_cells in worksheet.columns:
        letter = (
            column_cells[0]
            .column_letter
        )

        values = [
            str(cell.value or "")
            for cell in column_cells[
                : min(
                    len(column_cells),
                    150,
                )
            ]
        ]

        width = min(
            max(
                max(
                    (
                        len(value)
                        for value in values
                    ),
                    default=8,
                )
                + 2,
                10,
            ),
            50,
        )

        worksheet.column_dimensions[
            letter
        ].width = width

    for row in worksheet.iter_rows():
        for cell in row:
            if (
                isinstance(
                    cell.value,
                    str,
                )
                and len(cell.value) > 80
            ):
                cell.alignment = Alignment(
                    vertical="top",
                    wrap_text=True,
                )

workbook.save(
    PROCESSING_XLSX
)


print("Outputs:")
print("-", repo_path(TEXT_MANIFEST_CSV))
print("-", repo_path(PROCESSING_XLSX))
print("-", repo_path(PROCESSING_STATE_JSONL))
print(
    "-",
    repo_path(PROCESSED_TEXT_DIR)
    + "/<original_raw_basename>.txt",
)

if len(review_df):
    print()
    print(
        f"Rows requiring review: {len(review_df):,}"
    )

    display(
        review_df[
            [
                "file_id",
                "raw_file_name",
                "case_numbers",
                "file_format",
                "extraction_method",
                "processing_status",
                "review_reason",
                "fallback_reason",
                "error",
            ]
        ].head(25)
    )


Outputs:
- ec_cjeu/output/com_registry/com_registry_text_manifest.csv
- ec_cjeu/output/com_registry/com_registry_processing.xlsx
- ec_cjeu/logs/com_registry/processing_state.jsonl
- ec_cjeu/data/processed/com_registry/text/<original_raw_basename>.txt

Rows requiring review: 9


,file_id,raw_file_name,case_numbers,file_format,extraction_method,processing_status,review_reason,fallback_reason,error
166,COMFILE_277f18030f919b0c,COMFILE_277f18030f919b0c.pdf,AT.40512,pdf,mineru_ocr,review,mineru:encoding_corruption,pymupdf:encoding_corruption; pdftotext:encodin...,
359,COMFILE_5474d8c18620b632,COMFILE_5474d8c18620b632.pdf,AT.40608,pdf,mineru_ocr,review,mineru:too_little_text; mineru:low_text_per_page,pymupdf:too_little_text; pymupdf:low_text_per_...,
482,COMFILE_70ce8f8cd57fa8fb,COMFILE_70ce8f8cd57fa8fb.pdf,AT.35763,pdf,mineru_ocr,review,mineru:fragmented_text,pymupdf:too_little_text; pymupdf:low_text_per_...,
708,COMFILE_a283002ef6064a3b,COMFILE_a283002ef6064a3b.pdf,AT.37685,pdf,mineru_ocr,review,mineru:encoding_corruption,pymupdf:encoding_corruption; pdftotext:encodin...,
796,COMFILE_b2cf654b003de4ff,COMFILE_b2cf654b003de4ff.pdf,AT.35774,pdf,mineru_ocr,review,mineru:fragmented_text,pymupdf:too_little_text; pymupdf:low_text_per_...,
876,COMFILE_c3f2afaf5031b906,COMFILE_c3f2afaf5031b906.pdf,AT.35770,pdf,mineru_ocr,review,mineru:low_text_per_page,pymupdf:too_little_text; pymupdf:low_text_per_...,
902,COMFILE_cbb2a082639dc5f9,COMFILE_cbb2a082639dc5f9.pdf,AT.39839,pdf,mineru_ocr,review,mineru:encoding_corruption,pymupdf:encoding_corruption; pdftotext:encodin...,
930,COMFILE_d083829c67a44671,COMFILE_d083829c67a44671.pdf,AT.39839,pdf,mineru_ocr,review,mineru:encoding_corruption,pymupdf:encoding_corruption; pdftotext:encodin...,
999,COMFILE_dfea49a9aa1e9689,COMFILE_dfea49a9aa1e9689.pdf,AT.40452,pdf,mineru_ocr,review,mineru:low_text_per_page,pymupdf:low_text_per_page; pdftotext:low_text_...,


## 10. Technical validation for the Data Descriptor

Automated validation in this notebook is deliberately limited to reproducibility and
obvious extraction failures:

- the raw file must exist;
- its SHA-256 must match the acquisition manifest from notebook 06;
- the selected extraction must produce canonical UTF-8 text;
- the processed text file must remain reproducible through its recorded SHA-256.

For the frozen Scientific Data release, meaningful manual validation can then focus on:

- every document for which MinerU OCR was selected;
- every row marked `review` or `failed`;
- a stratified sample of ordinary native PDF and HTML extractions.

Manual inspection should verify that the processed text corresponds to the underlying
Commission document, substantial content has not been omitted, obvious corruption is
absent, and the reading order is broadly usable.

The final validation sample and resulting counts should be frozen and reported in the
Data Descriptor rather than encoded here as a large bespoke audit system.


## Reproducibility notes

This notebook is intended to sit under:

`ec_cjeu/code/01_commission/07_process_com_registry.ipynb`

Repository-root detection therefore recognizes the `code/01_commission` layout.

Core Python dependencies are:

- `pandas`
- `tqdm`
- `beautifulsoup4`
- `charset-normalizer`
- `PyMuPDF`
- `openpyxl`

Poppler's `pdftotext` is the preferred second native PDF extractor and MinerU is the
OCR fallback. MinerU is invoked explicitly with the `pipeline` backend and OCR mode,
with page-scaled timeout allowances for long scanned PDFs.

All persisted paths in manifests and JSONL are repository-relative and shown with the
`ec_cjeu/` prefix. The raw source files are never modified.


In [20]:
source /home/edik/projects/.venv/bin/activate

SyntaxError: invalid syntax (3138879986.py, line 1)